# Auric CV assignment: Colab training (B0, B1)

Run top to bottom on a GPU runtime (Runtime > Change runtime type > L4 or A100).
* Data is copied from Drive to `/content` and unzipped there (never trained from Drive).
* Every run folder (config, command, env, checkpoints, metrics) lives on Drive under `DRIVE_RUNS`,
  so a disconnect loses at most the current epoch. **To resume, just re-run the same training cell**:
  `train.py` continues from `<run>/train/weights/last.pt`.
* Tiles for B1 are rebuilt deterministically in `/content/work` after a restart (same seed, same files).

In [ ]:
# ---- edit these ----
DRIVE_DATA = "/content/drive/MyDrive/cv_dataset.zip"  # a .zip OR a folder containing train/ val/ classmap.txt
DRIVE_RUNS = "/content/drive/MyDrive/auric_runs"      # run folders persist here
REPO_URL   = "https://github.com/<you>/auric_cv_assignment.git"
BRANCH     = "main"
TOKEN_SECRET = "GH_TOKEN"  # Colab secret (key icon) holding a GitHub token; only needed if the repo is private
DATA_ROOT, WORK_DIR, REPO_DIR = "/content/data", "/content/work", "/content/repo"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_RUNS, exist_ok=True)
!nvidia-smi

In [ ]:
# Clone (or update) the repo
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get(TOKEN_SECRET)
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # public repo or no secret set
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt
import torch, ultralytics
print("torch", torch.__version__, "CUDA", torch.version.cuda, "GPU", torch.cuda.get_device_name(0), "ultralytics", ultralytics.__version__)

In [ ]:
# Copy dataset from Drive to local disk and unzip; generate data.yaml if missing
import os, shutil, subprocess, sys
from pathlib import Path
if not (Path(DATA_ROOT) / "_READY").exists():
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    if DRIVE_DATA.endswith(".zip"):
        subprocess.run(["cp", DRIVE_DATA, "/content/dataset.zip"], check=True)
        subprocess.run(["unzip", "-q", "/content/dataset.zip", "-d", DATA_ROOT], check=True)
        os.remove("/content/dataset.zip")
    else:
        shutil.copytree(DRIVE_DATA, DATA_ROOT)
    # if the archive had a wrapper folder, point DATA_ROOT at the folder that holds train/ and val/
    hits = [p.parent for p in Path(DATA_ROOT).rglob("classmap.txt") if (p.parent / "train").is_dir()]
    assert len(hits) == 1, f"expected one dataset root, found {hits}"
    if hits[0] != Path(DATA_ROOT):
        tmp = "/content/_data_tmp"; shutil.move(str(hits[0]), tmp); shutil.rmtree(DATA_ROOT); shutil.move(tmp, DATA_ROOT)
    (Path(DATA_ROOT) / "_READY").touch()
sys.path.insert(0, REPO_DIR)
from detlib.data import ensure_data_yaml
print(open(ensure_data_yaml(DATA_ROOT)).read())
for s in ("train", "val"):
    print(s, len(list((Path(DATA_ROOT) / s / "images").iterdir())), "images")

## B0: full image at 640 (reference). Re-run this cell to resume after a disconnect.

In [ ]:
!python train.py --config configs/b0.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS}

In [ ]:
!python eval.py --config configs/b0.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS} --ultra-crosscheck
!python analysis/pred_review.py --preds {DRIVE_RUNS}/b0_full640/eval/predictions.csv --name b0_full640 --data-root {DATA_ROOT} --out-root {DRIVE_RUNS}/figures

## B1: native-resolution 1024 tiles. Re-run this cell to resume after a disconnect.

In [ ]:
!python train.py --config configs/b1.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS} --work-dir {WORK_DIR}

In [ ]:
!python eval.py --config configs/b1.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS}
!python analysis/pred_review.py --preds {DRIVE_RUNS}/b1_tile1024/eval/predictions.csv --name b1_tile1024 --data-root {DATA_ROOT} --out-root {DRIVE_RUNS}/figures

## Summary and export (run folders without weights, to copy into the repo's runs/ and figures/)

In [ ]:
import json, pandas as pd, shutil
for run in ("b0_full640", "b1_tile1024"):
    p = Path(DRIVE_RUNS) / run / "eval" / "per_class.csv"
    if p.exists():
        print(run); print(pd.read_csv(p)[["class_name", "n_gt", "AP50_coco", "AP50_coco_ci95_lo", "AP50_coco_ci95_hi", "AP50_ultralytics"]].to_string(index=False))
!cd {DRIVE_RUNS} && zip -qr /content/runs_export.zip . -x "*.pt"
shutil.copy("/content/runs_export.zip", DRIVE_RUNS + "/../auric_runs_export.zip")
print("export:", DRIVE_RUNS + "/../auric_runs_export.zip  (weights stay in", DRIVE_RUNS, ")")